### Adaptive RAG

Import libraries

In [9]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import ChatPromptTemplate
from langchain.schema import StrOutputParser
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict


Step 2 — Prepare Sample Data

In [10]:
documents = [
    'Maize in Nigeria is affected by Fall Armyworm. Cause: insect infestation. Control: use resistant varieties, biological control with parasitoids.',
    'Cassava mosaic disease is caused by viruses spread by whiteflies. Control: plant resistant cassava varieties, remove infected plants.',
    'Rice blast disease in Nigeria is caused by fungus Magnaporthe oryzae. Control: fungicide application, crop rotation.',
    'Groundnut rosette disease is caused by viruses transmitted by aphids. Control: resistant varieties, vector control.',
    'Tomato leaf curl disease is caused by begomoviruses. Control: use virus-free seedlings, control whiteflies.'
]

Step 3 — Build llm model and Vector Store

In [11]:
# Define the LLM
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=20
)

docs = splitter.create_documents(documents)

# Create embeddings using OpenAI
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# Store in Chroma vector database
vectorstore = Chroma.from_documents(
    docs,
    embeddings,
    collection_name='agriculture_nigeria'
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


#### Step 4: Define the Adaptive RAG State Schema

This schema acts as a container for all workflow data.  
It ensures consistency across nodes in LangGraph.

**Fields:**
- **query**: the original user query string.  
- **retrieved_docs**: list of documents retrieved from the vector store.  
- **refined_query**: reformulated query if initial retrieval was insufficient.  
- **filtered_docs**: list of documents after filtering and summarization.  
- **answer**: final generated answer from the LLM.  
- **confidence**: confidence score for the generated answer.  


In [12]:
class AdaptiveState(TypedDict):
    query: str
    retrieved_docs: List[str]
    refined_query: str
    filtered_docs: List[str]
    answer: str
    confidence: float

Step 5a — Retrieval Node

In [13]:
def retrieved_node(state: AdaptiveState) -> Dict:
    '''
    Retrieval Node
    This node retrieves documents from the Chroma vector store
    based on the user query.

    Returns:
    - retrieved_docs: list of relevant documents.
    '''
    
    results = vectorstore.similarity_search(state['query'], k=5)
    docs = [r.page_content for r in results]
    
    return {
        'retrieved_docs': docs
    }
    
    
state = {'query': 'What causes cassava mosaic disease?'}
output = retrieved_node(state)
print(output)

{'retrieved_docs': ['Cassava mosaic disease is caused by viruses spread by whiteflies. Control: plant resistant cassava varieties, remove infected plants.', 'Cassava mosaic disease is caused by viruses spread by whiteflies. Control: plant resistant cassava varieties, remove infected plants.', 'Rice blast disease in Nigeria is caused by fungus Magnaporthe oryzae. Control: fungicide application, crop rotation.', 'Rice blast disease in Nigeria is caused by fungus Magnaporthe oryzae. Control: fungicide application, crop rotation.', 'Tomato leaf curl disease is caused by begomoviruses. Control: use virus-free seedlings, control whiteflies.']}


Step 5b — Critique Node

In [14]:
'''
Critique Node
    This node evaluates whether the retrieved documents
    are sufficient to answer the query.

    Improved Logic:
    - Instead of keyword search, we ask the LLM to judge
      if the docs contain enough relevant information.
    - This scales better for large PDF corpora.

    Returns:
    - sufficient: boolean indicating adequacy of retrieval.
'''

def critique_node(state: AdaptiveState) -> AdaptiveState:
    context = '\n'.join(state['retrieved_docs'])
    critique_prompt  = ChatPromptTemplate.from_messages([
        ('system', 'You are an evaluator of retrieval quality.'),
        ('user', 
         f'Query: {state["query"]}\n'
         f'Context: {context}\n\n'
         f'Does the context contain enough information to answer the query?\n'
         f'Reply only with True or False.'
         )
    ])
    
    # Build pipeline: Prompt → LLM → Parser
    critique_chain = critique_prompt | llm | StrOutputParser()
    
    # Pass in the state that contains the query and retrieved docs
    judgement = critique_chain.invoke(state).strip().lower()
    sufficient = 'true' in judgement
    
    # Convert the LLM output ("True"/"False") into a boolean
    state.update({'sufficient': sufficient})
    
    return state
    
state = {'query': 'What causes cassava mosaic disease?'}

state.update(retrieved_node(state))
state.update(critique_node(state))
print(state)

{'query': 'What causes cassava mosaic disease?', 'retrieved_docs': ['Cassava mosaic disease is caused by viruses spread by whiteflies. Control: plant resistant cassava varieties, remove infected plants.', 'Cassava mosaic disease is caused by viruses spread by whiteflies. Control: plant resistant cassava varieties, remove infected plants.', 'Rice blast disease in Nigeria is caused by fungus Magnaporthe oryzae. Control: fungicide application, crop rotation.', 'Rice blast disease in Nigeria is caused by fungus Magnaporthe oryzae. Control: fungicide application, crop rotation.', 'Tomato leaf curl disease is caused by begomoviruses. Control: use virus-free seedlings, control whiteflies.'], 'sufficient': True}


Step 5c — Refine Node

In [15]:
def refine_node(state: AdaptiveState) -> AdaptiveState:
    '''
    Step 6c: Refine Node
    If critique shows insufficient docs, this node reformulates
    the query to improve retrieval.

    Returns:
    - refined_query: reformulated query string.
    - retrieved_docs: new set of documents.
    '''
    
    if not state.get('sufficient', True):
        refined = state['query'] + ' Nigeria agriculture'
        results = vectorstore.similarity_search(refined, k=3)
        
        docs = [r.page_content for r in results]
        
        return {
            'refined_query': refined,
            'retrieved_docs': docs
        }
    return {}

state = {
    'query': 'Crop issues in Nigeria', 'retrieved_docs': [], 'sufficient': False
}
output = refine_node(state)
print(output)

{'refined_query': 'Crop issues in Nigeria Nigeria agriculture', 'retrieved_docs': ['Maize in Nigeria is affected by Fall Armyworm. Cause: insect infestation. Control: use resistant varieties, biological control with parasitoids.', 'Maize in Nigeria is affected by Fall Armyworm. Cause: insect infestation. Control: use resistant varieties, biological control with parasitoids.', 'Rice blast disease in Nigeria is caused by fungus Magnaporthe oryzae. Control: fungicide application, crop rotation.']}


Step 5d — Filter Node

In [17]:
def filter_node(state: AdaptiveState) -> AdaptiveState:
    '''
    Step 6d: Filter Node
    Filters retrieved documents to keep only those relevant
    to the query (e.g., control measures).

    Logic:
    - Ask the LLM to classify each doc as Cause, Symptom, or Control.
    - Normalize the output to lowercase.
    - Keep only docs classified exactly as "control".
    - Update the state with 'filtered_docs'.
    '''
    
    filtered = []
    
    for doc in state['retrieved_docs']:
        filter_prompt = ChatPromptTemplate.from_messages([
            ('system', 'Classify the text as Cause, Symptom, or Control.'),
            ('user', f'Text: {doc}\n\nReply only with one category.')
        ])
        
        # Build pipeline: Prompt → LLM → Parser
        filter_chain = filter_prompt | llm | StrOutputParser()
        
        # Run classification for this doc
        category = filter_chain.invoke({
            'doc': doc
        }).strip().lower()
        
        # Exact match check
        if 'control' in category:
            filtered.append(doc)

    # Update workflow state with filtered docs
    state.update({'filtered_docs': filtered})
    return state


state = {'query': 'What are control measures for cassava mosaic disease?'}
state = retrieved_node(state)
state = filter_node(state)
print(state['filtered_docs'])

[]


In [22]:
def filter_node(state: AdaptiveState) -> AdaptiveState:
    filtered = []
    
    for doc in state['retrieved_docs']:
        filter_prompt = ChatPromptTemplate.from_messages([
            ('system', 'Classify the text as Cause, Symptom, or Control. Reply ONLY with one word: Cause, Symptom, or Control.'),
            ('user', f'Text: {doc}')
        ])

        
        filter_chain = filter_prompt | llm | StrOutputParser()
        category = filter_chain.invoke({'doc': doc}).strip().lower()
        
        # Robust check
        if 'control' in category:
            filtered.append(doc)
    
    state.update({'filtered_docs': filtered})
    return state


state = {'query': 'What are control measures for cassava mosaic disease?'}
state = retrieved_node(state)       # fills in retrieved_docs
state = filter_node(state)          # filters them down
print(state['filtered_docs'])

[]
